# Zero-Shot Voice Cloning — Colab GPU Server

Run this notebook on a **Colab GPU runtime** to host the FastAPI cloning
backend and expose it via a public **ngrok** tunnel. Point the Streamlit UI
at the printed ngrok URL (`.../api/clone`).

## 1. Verify GPU & install dependencies

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
!pip install -q fastapi uvicorn streamlit python-multipart \
                 librosa noisereduce soundfile openai-whisper \
                 pyngrok --upgrade

# Install F5-TTS from source (the PyPI package may lag the repo).
!pip install -q git+https://github.com/SWivid/F5-TTS.git

print('Dependencies installed.')

## 2. Clone the pipeline source

In [ ]:
import os
import sys

WORKDIR = '/content/voice-cloner'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Working directory:', WORKDIR)

> If your repo is on GitHub, uncomment the next cell to clone it.
> Otherwise, upload the project files to the Colab file browser.

In [ ]:
# !git clone https://github.com/<USER>/<REPO>.git .

## 3. Start ngrok tunnel

In [ ]:
# Optional: embed your authtoken for persistent tunnels.
NGROK_AUTHTOKEN = ''  # <-- paste your token from https://dashboard.ngrok.com

from pyngrok import ngrok, conf

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

conf.get_default().monitor_thread = False  # avoid colab polling conflicts
tunnel = ngrok.connect(8000)
print('Public URL:', tunnel.public_url)

# Save for later use.
with open('ngrok_url.txt', 'w') as f:
    f.write(tunnel.public_url)

## 4. Launch the FastAPI backend on port 8000

In [ ]:
import subprocess
import threading


def run_server():
    """Start uvicorn in a background thread."""
    cmd = [
        sys.executable, '-m', 'uvicorn',
        'src.api.main:app',
        '--host', '0.0.0.0',
        '--port', '8000',
    ]
    proc = subprocess.Popen(cmd, cwd=WORKDIR)
    proc.wait()


thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print('Server started in background on port 8000.')

## 5. Verify

In [ ]:
import time
import requests

time.sleep(5)
base = open('ngrok_url.txt').read().strip()
try:
    r = requests.get(base + '/health', timeout=10)
    print('Health:', r.status_code, r.json())
except Exception as exc:
    print('Health check failed:', exc)

## 6. Use

- In `ui/app.py`, set the **API Endpoint URL** to:
  `https://<your-ngrok-subdomain>.ngrok-free.app/api/clone`
- Or call directly with `requests`:
```python
requests.post(
    ngrok_url + '/api/clone',
    files={'ref_audio': ('ref.wav', open('ref.wav','rb'), 'audio/wav')},
    data={'target_text': 'Hello, this is a cloned voice.'},
).content  # -> WAV bytes
```